# Grammar Scoring Engine - v4 (stronger audio representations)

## Where v3 left us

Your executed v3 notebook gave these out-of-fold (CV) numbers:

| Model family | CV RMSE |
|---|---|
| Whisper-small encoder embedding (Ridge / SVR) | **0.566 / 0.573** |
| Tabular features (XGBoost) | 0.611 |
| Tabular features (ExtraTrees / Ridge) | 0.644 / 0.664 |
| Sentence-embedding of the transcript | 0.93 - 0.94 |
| TF-IDF of the transcript | 1.05 |
| **Stack of everything** | **0.5275** |

The stacker gave about 70% of its weight to the Whisper-embedding models and the XGBoost model.
The leaderboard moved from 0.64 to 0.39, and the top competitors are at about 0.30.

## What this tells us

1. **The audio carries most of the signal.** A raw audio embedding with a plain linear model beats every text-based model by a wide margin.
   The transcript is a weak signal (Whisper cleans up speech and loses the errors).
2. **v3 used only one layer of one audio model** (the last layer of Whisper-small, mean + std pooled).
   Last layers of speech models are specialised for the model's own training task (transcription). Middle layers usually hold the pronunciation, fluency and prosody information that rater scores depend on.
3. **Only one audio model was used.** Different speech models (Whisper, WavLM, HuBERT) capture different things; combining them is the cheapest large gain.

## What v4 changes

| Step | v3 | v4 |
|---|---|---|
| Audio embeddings | Whisper-small, last layer only | **4 backbones x several layers each**: Whisper-small, Whisper-medium, WavLM-large, HuBERT-large |
| Choosing a layer | none | **Layer sweep**: one Ridge model per layer, keep the best layers per backbone |
| Combining audio models | only inside the stacker | **Fusion models**: one Ridge / SVR on the best layer of every backbone (+ tabular + text) |
| Test distribution | not checked | **Adversarial validation** (can a classifier tell train from test?) |
| Submission IDs | mapped by exact name, unmatched rows silently became 0 | **ID reconciliation** with a loud report, never silent zeros |
| Post-processing | none | optional **isotonic calibration**, used only if it helps in CV |
| Error analysis | none | RMSE and bias **per label band** |
| Feature cache | rebuilt | **Reuses your v3 cache** (no re-transcription), new embeddings are cached separately |

## Important: check your submission file first

In your v3 notebook, `sample_submission.csv` has **204 rows** but `test.csv` has **216 rows**.
Your `submission_v3.csv` printed `mean 0.40`, `25% = 50% = 75% = 0`, which means most rows were filled with the fallback value `0.0` because the IDs did not match.
Section 4 of this notebook prints exactly how many IDs match and section 17 will not write a misleading file.
**Please tell me / check which file you actually uploaded to Kaggle** (the 204-row one or the 216-row one), because that changes how to read your 0.39.

## Runtime expectations (rough, not measured)

* Loading cached v3 features: a minute or two.
* Extracting new embeddings on a T4: roughly 10-15 minutes per backbone the first time (cached afterwards). Set `"enabled": False` for a backbone in the config cell to skip it.
* Modeling: several minutes.

## Honest expectations

Stronger audio embeddings and fusion are the best-supported next step given your own results, but I cannot promise the 0.30 level.
The CV numbers and the leaderboard do not match exactly (your CV was 0.53, the leaderboard 0.39), so judge changes by the **relative CV improvement** over the 0.5275 baseline.

## 1. Install dependencies

**What the cell does**
* Installs the Python packages: `openai-whisper` (speech model), `librosa` (audio loading), `transformers` (WavLM / HuBERT / CoLA / GPT-2), `sentence-transformers` (transcript embeddings), `xgboost`, `language_tool_python` (grammar checker) and `nltk`.
* Installs Java (`default-jre-headless`), which LanguageTool needs.

**Why it is still needed even with the v3 cache**
The cache covers every file, so the heavy extraction models are normally never loaded. The packages are still imported at the top of the notebook, and the lazy loader falls back to them if any cache file is missing.

**What to look for**
No errors. Warnings about dependency versions are normal in Colab.

In [ ]:
!pip -q install openai-whisper librosa nltk xgboost soundfile language_tool_python transformers sentence-transformers sentencepiece
!apt-get -qq install -y default-jre-headless > /dev/null

## 2. Mount Google Drive

**What the cell does**
Connects your Google Drive so the notebook can read `train.csv`, `test.csv`, the audio folders and the cached v3 features, and write the new caches and submission files back.

**What to look for**
`Mounted at /content/drive` and an authorisation popup the first time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Imports and configuration

**What the cell does**
* Imports every library used later.
* Fixes the random seeds so reruns give the same folds.
* Defines the **configuration**, the only place you normally need to edit:
  * `WHISPER_SIZE = "small"` - must stay `"small"` to reuse the v3 transcript cache (the cache folder name contains it).
  * `N_SPLITS`, `N_REPEATS` - repeated 5-fold cross-validation, same as v3, so numbers are comparable.
  * `TOP_K_LAYERS = 2` - how many of the best layers per audio backbone are kept after the layer sweep.
  * `V3_STACK_CV = 0.5275` - your v3 stacked CV RMSE, used for a direct before/after comparison.
  * `BACKBONES` - the audio models to extract embeddings from, and **which layers** to pool.
    * `kind: "whisper"` models use the Whisper **encoder**; a layer number `L` means "output of encoder block L".
    * `kind: "hf"` models come from Hugging Face; a layer number `L` indexes `hidden_states` (0 = input projection, 1..N = transformer layers).
    * Set `"enabled": False` to skip a backbone and save time.
    * To try an even bigger model, add `{"name": "whisper_large", "kind": "whisper", "id": "large-v3", "layers": [16, 20, 24, 28, 32], "enabled": True}` (needs more GPU memory).

**What to look for**
`Device: cuda` and your GPU name. Without a GPU the extraction would be very slow.

In [ ]:
import os, re, gc, math, pickle, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import nltk
import torch
import whisper
from tqdm.auto import tqdm

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression, LogisticRegression
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import (RepeatedStratifiedKFold, RepeatedKFold, StratifiedKFold,
                                     GridSearchCV, KFold, cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error, roc_auc_score
from scipy.stats import pearsonr, spearmanr
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ------------------------- configuration -------------------------
SEED = 42
WHISPER_SIZE = "small"        # keep "small" to reuse the v3 transcript/feature cache
N_SPLITS = 5
N_REPEATS = 3
TOP_K_LAYERS = 2              # best layers kept per backbone after the layer sweep
V3_STACK_CV = 0.5275          # your v3 stacked CV RMSE (for comparison)

BACKBONES = [
    {"name": "whisper_small",   "kind": "whisper", "id": "small",
     "layers": [6, 8, 10, 12],          "enabled": True},
    {"name": "whisper_medium",  "kind": "whisper", "id": "medium",
     "layers": [12, 16, 20, 24],        "enabled": True},
    {"name": "wavlm_large",     "kind": "hf", "id": "microsoft/wavlm-large",
     "layers": [8, 12, 16, 20, 24],     "enabled": True},
    {"name": "hubert_large_ft", "kind": "hf", "id": "facebook/hubert-large-ls960-ft",
     "layers": [8, 12, 16, 20, 24],     "enabled": True},
]
# ------------------------------------------------------------------

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 4. NLTK resources

**What the cell does**
Downloads the tokenizer and part-of-speech tagger data used by the text features (word/sentence splitting and POS tags).

**What to look for**
`NLTK ready`.

In [ ]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
print("NLTK ready")

## 5. Paths, data, and the submission-ID check

**What the cell does**
* Defines all paths. `CACHE_DIR` points to your **existing v3 cache** (`feature_cache_v3_small`) so transcripts and tabular features are not recomputed. New embeddings go to a separate folder, `emb_cache_v4`.
* Loads `train.csv`, `test.csv`, `sample_submission.csv` and checks that every audio file exists.
* **New:** compares the IDs in `test.csv` with the IDs in `sample_submission.csv`, exactly and after normalising (extension removed, lower-cased).

**Why this matters**
Last time `sample_submission` had 204 rows but `test.csv` had 216, and most of the 204-row submission ended up as `0.0` because the names did not match. This cell tells you immediately how big the mismatch is and shows examples from both files so you can see *why* they differ.

**What to look for**
* `Exact ID overlap` and `Overlap after normalising` - ideally both equal the number of sample rows.
* If they are far below 204, read the example IDs printed below the counts. Section 17 handles this safely, but you should know about it.

In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/data")

TRAIN_CSV = DATA_ROOT / "train.csv"
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
TRAIN_AUDIO_DIR = DATA_ROOT / "train"
TEST_AUDIO_DIR = DATA_ROOT / "test"

CACHE_DIR = DATA_ROOT / f"feature_cache_v3_{WHISPER_SIZE}"      # reused from v3
EMB_CACHE_DIR = DATA_ROOT / "emb_cache_v4"                      # new, per backbone
CACHE_DIR.mkdir(parents=True, exist_ok=True)
EMB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("v3 cache files found:", len(list(CACHE_DIR.glob("*.pkl"))))

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_CSV)
id_col, label_col = sample_sub.columns[0], sample_sub.columns[1]

print("Train:", train_df.shape, "| Test:", test_df.shape, "| Sample sub:", sample_sub.shape)
print(train_df["label"].describe())


def check_audio_files(df, audio_dir, name):
    missing = [f for f in df["filename"].astype(str) if not (Path(audio_dir) / f).exists()]
    print(f"{name}: {len(df) - len(missing)}/{len(df)} files found")
    return missing

_ = check_audio_files(train_df, TRAIN_AUDIO_DIR, "TRAIN")
_ = check_audio_files(test_df, TEST_AUDIO_DIR, "TEST")


# ---------------- submission ID diagnostic ----------------
def norm_id(s):
    return Path(str(s)).stem.lower().strip()

sample_ids_raw = sample_sub[id_col].astype(str).tolist()
test_ids_raw = test_df["filename"].astype(str).tolist()
exact_overlap = len(set(sample_ids_raw) & set(test_ids_raw))
test_norm = {norm_id(f) for f in test_ids_raw}
stem_overlap = sum(norm_id(s) in test_norm for s in sample_ids_raw)

print(f"\nSample-submission rows: {len(sample_ids_raw)} | test.csv rows: {len(test_ids_raw)}")
print(f"Exact ID overlap:           {exact_overlap}")
print(f"Overlap after normalising:  {stem_overlap}")
print("Example IDs in sample_submission:", sample_ids_raw[:5])
print("Example IDs in test.csv:         ", test_ids_raw[:5])

## 6. Lazy loading of the extraction models

**What the cell does**
Defines `ensure_extraction_models()`, which loads Whisper (for transcripts), LanguageTool, the CoLA grammar-acceptability model and GPT-2 **only the first time a file is missing from the v3 cache**.

**Why**
In v3 these models were always loaded (several GB, a few minutes). Since your cache already holds every file, v4 skips them completely and starts modeling much sooner. If you ever add new audio files, they will be processed automatically.

**What to look for**
Nothing is printed now. The models load later only if needed.

In [ ]:
EXTRACTION_READY = False
LT_OK = False
whisper_model = lt_tool = None
cola_tok = cola_model = gpt_tok = gpt_model = None


def ensure_extraction_models():
    global EXTRACTION_READY, LT_OK, whisper_model, lt_tool, cola_tok, cola_model, gpt_tok, gpt_model
    if EXTRACTION_READY:
        return
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              GPT2LMHeadModel, GPT2TokenizerFast)
    print("Loading extraction models (only needed for files missing from the cache) ...")
    whisper_model = whisper.load_model(WHISPER_SIZE, device=DEVICE)
    try:
        import language_tool_python
        lt_tool = language_tool_python.LanguageTool("en-US")
        lt_tool.check("This is a test sentence.")
        LT_OK = True
    except Exception as e:
        print("LanguageTool NOT available, its features will be skipped:", e)
    cola_tok = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
    cola_model = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA").to(DEVICE).eval()
    gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
    gpt_model = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()
    EXTRACTION_READY = True
    print("Extraction models loaded.")

## 7. Feature functions (unchanged from v3)

The next four cells are the **same feature code as v3**. They are kept so the notebook is complete and can process new files, and so the cached features stay consistent.
They are only *called* for files that are not in the cache.

### 7a. Whisper confidence and timing features

**What it computes** (from Whisper's word-level output):
* Word probability statistics (mean, std, min, 10th/25th percentile, fraction below 0.5 / 0.3). Low confidence is a proxy for unclear or erroneous speech.
* Segment log-probability, compression ratio and no-speech probability (flags hallucinated or repetitive output).
* Timing: speech rate, articulation rate, phonation ratio, counts and lengths of pauses (>= 0.25 s, 0.5 s, 1 s), leading/trailing silence.
* Disfluency: adjacent repeated words ("I I went"), filler tokens ("um", "uh").

In [ ]:
# ---------------- 1. Whisper confidence + timing ----------------
FILLERS = {"uh", "um", "erm", "hmm", "er", "ah", "uhm"}


def whisper_features(segments, duration):
    f = {"n_segments": len(segments)}

    if segments:
        lp = np.array([s.get("avg_logprob", np.nan) for s in segments], dtype=float)
        cr = np.array([s.get("compression_ratio", np.nan) for s in segments], dtype=float)
        ns = np.array([s.get("no_speech_prob", np.nan) for s in segments], dtype=float)
        f["seg_logprob_mean"] = np.nanmean(lp)
        f["seg_logprob_min"] = np.nanmin(lp)
        f["seg_compress_max"] = np.nanmax(cr)
        f["seg_nospeech_max"] = np.nanmax(ns)

    words = []
    for s in segments:
        for w in (s.get("words") or []):
            if w.get("start") is not None and w.get("end") is not None:
                words.append(w)
    if not words:
        return f

    probs = np.array([float(w.get("probability", np.nan)) for w in words])
    starts = np.array([float(w["start"]) for w in words])
    ends = np.array([float(w["end"]) for w in words])
    norm = [re.sub(r"[^a-z']", "", w["word"].lower()) for w in words]
    n = len(words)

    f["wp_mean"] = np.nanmean(probs)
    f["wp_std"] = np.nanstd(probs)
    f["wp_min"] = np.nanmin(probs)
    f["wp_p10"] = np.nanpercentile(probs, 10)
    f["wp_p25"] = np.nanpercentile(probs, 25)
    f["wp_frac_lt_05"] = np.nanmean(probs < 0.5)
    f["wp_frac_lt_03"] = np.nanmean(probs < 0.3)

    word_dur = np.clip(ends - starts, 0, None)
    gaps = np.clip(starts[1:] - ends[:-1], 0, None) if n > 1 else np.array([])
    pauses = gaps[gaps >= 0.25]

    f["wd_mean"] = word_dur.mean()
    f["wd_std"] = word_dur.std()
    f["lead_silence"] = starts[0]
    f["trail_silence"] = max(0.0, duration - ends[-1])
    f["speech_span"] = ends[-1] - starts[0]
    f["speech_rate"] = n / max(duration, 1e-6)
    f["articulation_rate"] = n / max(word_dur.sum(), 1e-6)
    f["phonation_ratio"] = word_dur.sum() / max(duration, 1e-6)
    f["n_pauses_025"] = len(pauses)
    f["n_pauses_050"] = int((gaps >= 0.5).sum())
    f["n_pauses_100"] = int((gaps >= 1.0).sum())
    f["pauses_per_word"] = len(pauses) / n
    f["pause_mean"] = pauses.mean() if len(pauses) else 0.0
    f["pause_max"] = pauses.max() if len(pauses) else 0.0
    f["pause_total_ratio"] = pauses.sum() / max(duration, 1e-6)

    f["adj_repeat_words"] = sum(1 for a, b in zip(norm[:-1], norm[1:]) if a and a == b)
    f["adj_repeat_ratio"] = f["adj_repeat_words"] / n
    f["filler_tokens"] = sum(1 for w in norm if w in FILLERS)
    return f

### 7b. Text and POS features

**What it computes** (from the Whisper transcript):
* Part-of-speech ratios (nouns, verbs, tenses, modals, prepositions ...), verb/noun ratio, share of past-tense verbs.
* Lexical variety (unique-word ratio, root type-token ratio), average word length, long-word ratio.
* Subordination rate (because, although, which ...), filler phrases ("like", "you know").
* Sentence structure: number of sentences, mean/std/max length, fraction of very short and very long sentences.

These describe style and complexity, not correctness. The next cell adds correctness.

In [ ]:
# ---------------- 2. Text / POS features ----------------
POS_TAGS = ["NN", "NNS", "NNP", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "MD", "JJ", "JJR", "JJS",
            "RB", "PRP", "PRP$", "DT", "IN", "CC", "TO", "WDT", "WP", "WRB", "EX", "RP", "CD"]
SUBORD = {"because", "although", "though", "while", "whereas", "if", "when", "since", "unless",
          "which", "who", "whom", "whose", "that", "after", "before", "until"}
FILLER_PHRASES = ["like", "actually", "basically", "you know", "i mean", "kind of", "sort of"]


def text_features(text, duration):
    f = {}
    if not isinstance(text, str) or not text.strip():
        f["n_words"] = 0
        return f

    toks = word_tokenize(text)
    tagged = pos_tag(toks)
    alpha = [(w, t) for w, t in tagged if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", w)]
    n = len(alpha)
    f["n_words"] = n
    if n == 0:
        return f

    words = [w.lower() for w, _ in alpha]
    tags = [t for _, t in alpha]
    tc = Counter(tags)

    for t in POS_TAGS:
        f[f"pos_{t}"] = tc.get(t, 0) / n

    nouns = sum(v for k, v in tc.items() if k.startswith("NN"))
    verbs = sum(v for k, v in tc.items() if k.startswith("VB"))
    adjs = sum(v for k, v in tc.items() if k.startswith("JJ"))
    advs = sum(v for k, v in tc.items() if k.startswith("RB"))
    f["noun_ratio"] = nouns / n
    f["verb_ratio"] = verbs / n
    f["adj_ratio"] = adjs / n
    f["adv_ratio"] = advs / n
    f["verb_noun_ratio"] = verbs / max(nouns, 1)
    f["adj_noun_ratio"] = adjs / max(nouns, 1)
    f["past_verb_share"] = tc.get("VBD", 0) / max(verbs, 1)
    f["pos_diversity"] = len(set(tags)) / n

    trans = list(zip(tags[:-1], tags[1:]))
    f["pos_bigram_diversity"] = len(set(trans)) / max(len(trans), 1)

    uniq = len(set(words))
    f["unique_ratio"] = uniq / n
    f["root_ttr"] = uniq / math.sqrt(n)
    wl = np.array([len(w.replace("'", "")) for w in words])
    f["avg_word_len"] = wl.mean()
    f["long_word_ratio"] = float((wl >= 7).mean())
    f["subord_rate"] = sum(w in SUBORD for w in words) / n

    wc = Counter(words)
    f["repeated_word_ratio"] = sum(c - 1 for c in wc.values() if c > 1) / n

    low = text.lower()
    f["filler_phrase_count"] = sum(len(re.findall(r"\b" + re.escape(p) + r"\b", low)) for p in FILLER_PHRASES)
    f["words_per_sec_text"] = n / max(duration, 1e-6)

    sents = sent_tokenize(text)
    lens = [len(re.findall(r"[A-Za-z']+", s)) for s in sents]
    lens = [l for l in lens if l > 0]
    f["n_sentences"] = len(lens)
    if lens:
        f["sent_len_mean"] = float(np.mean(lens))
        f["sent_len_std"] = float(np.std(lens))
        f["sent_len_max"] = float(np.max(lens))
        f["sent_len_min"] = float(np.min(lens))
        f["short_sent_frac"] = float(np.mean(np.array(lens) < 4))
        f["long_sent_frac"] = float(np.mean(np.array(lens) >= 15))
    return f

### 7c. Grammar-error features (LanguageTool, CoLA, GPT-2)

**What it computes**
* **LanguageTool**: number of detected issues (total, "core" excluding punctuation/casing/style that Whisper decides, and per 100 words), split by category (grammar, typos, confused words, ...).
* **CoLA** (a RoBERTa model trained to judge whether a sentence is grammatically acceptable): mean, minimum, std and fraction of sentences below 0.5.
* **GPT-2 perplexity**: mean, max, median, std of per-sentence negative log-likelihood. Ungrammatical or unusual text is less probable.

These are the only features that look directly at grammatical correctness.

In [ ]:
# ---------------- 3. Grammar-error features ----------------
LT_CATS = ["GRAMMAR", "TYPOS", "CONFUSED_WORDS", "SEMANTICS", "REDUNDANCY", "COMPOUNDING",
           "COLLOCATIONS", "NONSTANDARD_PHRASES", "MISC"]
LT_IGNORE_FOR_CORE = {"PUNCTUATION", "CASING", "TYPOGRAPHY", "STYLE"}   # Whisper decides these, not the speaker


def languagetool_features(text, n_words):
    if not LT_OK or not text.strip():
        return {}
    try:
        matches = lt_tool.check(text)
    except Exception:
        return {}
    cats = [str(getattr(m, "category", "UNK")).upper() for m in matches]
    rules = [str(getattr(m, "ruleId", "")) for m in matches]
    nw = max(n_words, 1)
    core = [c for c in cats if c not in LT_IGNORE_FOR_CORE]
    f = {
        "lt_total": len(matches),
        "lt_total_per100": 100.0 * len(matches) / nw,
        "lt_core": len(core),
        "lt_core_per100": 100.0 * len(core) / nw,
        "lt_unique_rules": len(set(rules)),
    }
    for c in LT_CATS:
        k = sum(1 for x in cats if x == c)
        f[f"lt_{c.lower()}"] = k
        f[f"lt_{c.lower()}_per100"] = 100.0 * k / nw
    return f


@torch.no_grad()
def cola_scores(sentences):
    if not sentences:
        return []
    enc = cola_tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    probs = torch.softmax(cola_model(**enc).logits, dim=-1)[:, 1]   # label 1 = acceptable
    return probs.float().cpu().tolist()


@torch.no_grad()
def sentence_nlls(sentences):
    out = []
    bos = torch.full((1, 1), gpt_tok.bos_token_id, dtype=torch.long, device=DEVICE)
    for s in sentences:
        ids = gpt_tok(s, return_tensors="pt", truncation=True, max_length=200).input_ids.to(DEVICE)
        ids = torch.cat([bos, ids], dim=1)
        logits = gpt_model(ids).logits[0, :-1]
        loss = torch.nn.functional.cross_entropy(logits, ids[0, 1:], reduction="mean")
        out.append(loss.item())
    return out


def lm_grammar_features(text):
    f = {}
    sents = [s for s in sent_tokenize(text) if len(re.findall(r"[A-Za-z']+", s)) >= 3]
    if not sents:
        return f
    cs = np.array(cola_scores(sents))
    f["cola_mean"] = cs.mean()
    f["cola_min"] = cs.min()
    f["cola_std"] = cs.std()
    f["cola_frac_lt_05"] = float((cs < 0.5).mean())
    ns = np.array(sentence_nlls(sents))
    f["gpt_nll_mean"] = ns.mean()
    f["gpt_nll_max"] = ns.max()
    f["gpt_nll_median"] = float(np.median(ns))
    f["gpt_nll_std"] = ns.std()
    return f

### 7d. Small acoustic feature set

**What it computes**
Duration, energy (RMS), zero-crossing rate, spectral centroid, 13 MFCC means, pitch statistics (mean, std, 5-95% range) and the fraction of the clip that is voiced.
This is deliberately small: the v2 notebook used about 75 acoustic features that were mostly noise. The rich audio information now comes from the embeddings in section 10.

In [ ]:
# ---------------- 4. Small acoustic set + 5. Whisper encoder embedding ----------------
def acoustic_features(y, sr=16000):
    f = {"audio_duration": len(y) / sr}
    rms = librosa.feature.rms(y=y)[0]
    f["rms_mean"] = rms.mean()
    f["rms_std"] = rms.std()
    f["zcr_mean"] = librosa.feature.zero_crossing_rate(y)[0].mean()
    f["centroid_mean"] = librosa.feature.spectral_centroid(y=y, sr=sr)[0].mean()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        f[f"mfcc_{i + 1}_mean"] = mfcc[i].mean()
    try:
        f0 = librosa.yin(y, fmin=70, fmax=400, sr=sr)
        f0 = f0[np.isfinite(f0)]
        f0 = f0[(f0 >= 70) & (f0 <= 400)]
        if len(f0):
            f["pitch_mean"] = f0.mean()
            f["pitch_std"] = f0.std()
            f["pitch_range"] = np.percentile(f0, 95) - np.percentile(f0, 5)
    except Exception:
        pass
    intervals = librosa.effects.split(y, top_db=30)
    f["voiced_fraction"] = sum(e - s for s, e in intervals) / max(len(y), 1)
    return f

## 8. Per-file extraction with caching

**What the cell does**
* `extract_all(path)` runs the full v3 feature pipeline for one clip: load audio, transcribe with Whisper (word timestamps, greedy decoding), and compute all feature groups. It calls `ensure_extraction_models()` first (lazy load).
* `process_dataset(...)` loops over a dataframe, **loads each file's record from the v3 cache if it exists**, otherwise extracts and saves it. A file that fails keeps its row with empty features, so row order and counts always match the CSV.

**Note on the old embedding field**
The v3 cache also stored a last-layer Whisper-small embedding (`emb`). It is no longer used; section 10 replaces it with the multi-layer version. New cache records store `emb = None`.

**What to look for**
Nothing yet; this only defines functions.

In [ ]:
def extract_all(path):
    ensure_extraction_models()
    y, sr = librosa.load(str(path), sr=16000, mono=True)
    if len(y) == 0:
        raise ValueError("Empty audio")
    duration = len(y) / sr

    result = whisper_model.transcribe(
        y, language="en", fp16=(DEVICE == "cuda"), temperature=0.0,
        condition_on_previous_text=False, word_timestamps=True, verbose=None)
    text = result.get("text", "").strip()
    segments = result.get("segments", [])

    feats = {}
    feats.update(acoustic_features(y, sr))
    feats.update(whisper_features(segments, duration))
    feats.update(text_features(text, duration))
    n_words = feats.get("n_words", 0)
    feats.update(languagetool_features(text, n_words))
    feats.update(lm_grammar_features(text))
    return feats, None, text


def process_dataset(df, audio_dir, split_name):
    rows, failed, n_cached = [], [], 0
    for fname in tqdm(df["filename"].astype(str).tolist(), desc=split_name):
        cache_path = CACHE_DIR / f"{split_name}_{Path(fname).stem}.pkl"
        try:
            if cache_path.exists():
                with open(cache_path, "rb") as fh:
                    rec = pickle.load(fh)
                n_cached += 1
            else:
                feats, emb, text = extract_all(Path(audio_dir) / fname)
                rec = {"feats": feats, "emb": emb, "text": text}
                with open(cache_path, "wb") as fh:
                    pickle.dump(rec, fh)
        except Exception as e:
            print(f"FAILED: {fname} -> {e}")
            failed.append((fname, str(e)))
            rec = {"feats": {}, "emb": None, "text": ""}
        rows.append(rec)
    print(f"{split_name}: {n_cached} from cache, {len(df) - n_cached - len(failed)} newly extracted, {len(failed)} failed")
    return rows, failed

### 8b. Run the extraction (loads from cache)

**What the cell does**
Calls `process_dataset` for the training and test sets.

**What to look for**
`train: 769 from cache, 0 newly extracted, 0 failed` and `test: 216 from cache, 0 newly extracted, 0 failed`. If it says some files are "newly extracted", the cache was incomplete and the lazy loader will take a few minutes to load the models.

In [ ]:
train_rows, train_failed = process_dataset(train_df, TRAIN_AUDIO_DIR, "train")
test_rows, test_failed = process_dataset(test_df, TEST_AUDIO_DIR, "test")

## 9. Assemble tabular features and transcript embeddings

**What the cell does**
* Builds the label vector `y` and the transcript arrays (`texts_tr`, `texts_te`).
* Builds the **tabular feature matrix** from all cached feature dictionaries, keeps numeric columns that are at least half filled and not constant, fills gaps with the **training median**, and **winsorises** (clips) at the training 0.5% / 99.5% percentiles so outliers cannot dominate Ridge/SVR. Test uses the training statistics only.
* Encodes each transcript with a sentence-transformer (`all-mpnet-base-v2`). This was a weak model in v3 (RMSE about 0.93) but it adds a little diversity in the stack, so it is kept.

**What to look for**
* `Empty transcripts` (v3 had 4 train / 1 test, meaning Whisper heard nothing; those rows rely on imputed features and the audio embeddings).
* `Tabular features: (769, 132)` as in v3.

In [ ]:
y = train_df["label"].astype(float).values
assert not np.isnan(y).any(), "NaN in labels"

texts_tr = np.array([r["text"] for r in train_rows], dtype=object)
texts_te = np.array([r["text"] for r in test_rows], dtype=object)
print("Empty transcripts  train/test:", int((texts_tr == "").sum()), "/", int((texts_te == "").sum()))

tab_tr = pd.DataFrame([r["feats"] for r in train_rows])
tab_te = pd.DataFrame([r["feats"] for r in test_rows])

FEATURE_COLUMNS = [c for c in tab_tr.columns
                   if pd.api.types.is_numeric_dtype(tab_tr[c])
                   and tab_tr[c].notna().mean() >= 0.5
                   and tab_tr[c].nunique(dropna=True) > 1]
tab_te = tab_te.reindex(columns=FEATURE_COLUMNS)
tab_tr = tab_tr[FEATURE_COLUMNS]

tab_tr = tab_tr.replace([np.inf, -np.inf], np.nan)
tab_te = tab_te.replace([np.inf, -np.inf], np.nan)
train_medians = tab_tr.median()
tab_tr = tab_tr.fillna(train_medians)
tab_te = tab_te.fillna(train_medians)

lo, hi = tab_tr.quantile(0.005), tab_tr.quantile(0.995)
tab_tr = tab_tr.clip(lo, hi, axis=1)
tab_te = tab_te.clip(lo, hi, axis=1)

X_tab = tab_tr.values.astype(np.float64)
X_tab_te = tab_te.values.astype(np.float64)
print("Tabular features:", X_tab.shape, "| test:", X_tab_te.shape)

from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
E_tx = st_model.encode(texts_tr.tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
E_tx_te = st_model.encode(texts_te.tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
del st_model
gc.collect()
print("Text embedding:", E_tx.shape)

## 10. Multi-layer audio embeddings (the main upgrade)

### 10a. Extraction functions

**What the cell does**
Defines how one audio clip becomes a set of **fixed-length vectors, one per chosen layer**:

* **Chunking:** clips are processed in 30-second chunks (Whisper's native window; also keeps WavLM/HuBERT memory low). A tiny tail under 1 second is ignored.
* **Pooling:** for each layer, the hidden states of all valid frames (padding excluded for Whisper) are summarised by **mean and standard deviation** over time. The mean captures the typical "voice quality"; the std captures how much it varies (prosody, hesitation). Result per layer: a vector of size `2 x hidden_dim`.
* **Whisper backbones:** forward hooks capture the output of encoder block `L`. The decoder is deleted after loading to save memory.
* **Hugging Face backbones (WavLM, HuBERT):** `output_hidden_states=True`, take `hidden_states[L]`. The model's own feature extractor normalises the waveform as that model expects.
* **Mixed precision** (`autocast` fp16) on GPU for speed and memory.
* `extract_backbone(bb)` loops over all train then test clips, **caches one `.npy` per clip per backbone** in `emb_cache_v4` (so an interrupted run resumes), builds a `(N, n_layers, 2*dim)` array, and fills any failed clip with the training mean.

**Why several layers**
Which layer works best is an empirical question. The layer sweep in section 12 measures it instead of guessing.

**What to look for**
Nothing yet; only definitions.

In [ ]:
def autocast_ctx():
    return torch.autocast(device_type=("cuda" if DEVICE == "cuda" else "cpu"),
                          dtype=torch.float16, enabled=(DEVICE == "cuda"))


def finalize_pool(sums, sqs, total, layers):
    out = []
    for l in layers:
        mean = sums[l] / total
        std = (sqs[l] / total - mean ** 2).clamp_min(0).sqrt()
        out.append(torch.cat([mean, std]).cpu().numpy())
    return np.stack(out).astype(np.float16)


@torch.no_grad()
def whisper_layer_embedding(state, y_audio, layers):
    model, n_mels = state["model"], state["n_mels"]
    chunk = 30 * 16000
    captured, hooks = {}, []
    for l in layers:
        def hook(mod, inp, out, l=l):
            captured[l] = out[0] if isinstance(out, tuple) else out
        hooks.append(model.encoder.blocks[l - 1].register_forward_hook(hook))
    sums = {l: 0.0 for l in layers}
    sqs = {l: 0.0 for l in layers}
    total = 0
    try:
        for s in range(0, len(y_audio), chunk):
            seg = y_audio[s:s + chunk]
            if s > 0 and len(seg) < 16000:
                continue
            n_valid = max(1, int(math.ceil(len(seg) / chunk * 1500)))
            mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(seg), n_mels=n_mels).to(DEVICE)
            with autocast_ctx():
                model.encoder(mel.unsqueeze(0))
            for l in layers:
                h = captured[l][0, :n_valid].float()
                sums[l] = sums[l] + h.sum(0)
                sqs[l] = sqs[l] + (h ** 2).sum(0)
            total += n_valid
    finally:
        for h in hooks:
            h.remove()
    return finalize_pool(sums, sqs, total, layers)


@torch.no_grad()
def hf_layer_embedding(state, y_audio, layers):
    model, fe = state["model"], state["fe"]
    chunk = 30 * 16000
    sums = {l: 0.0 for l in layers}
    sqs = {l: 0.0 for l in layers}
    total = 0
    for s in range(0, len(y_audio), chunk):
        seg = y_audio[s:s + chunk]
        if s > 0 and len(seg) < 16000:
            continue
        inputs = fe(seg, sampling_rate=16000, return_tensors="pt")
        iv = inputs["input_values"].to(DEVICE)
        with autocast_ctx():
            hs = model(iv, output_hidden_states=True).hidden_states
        for l in layers:
            h = hs[l][0].float()
            sums[l] = sums[l] + h.sum(0)
            sqs[l] = sqs[l] + (h ** 2).sum(0)
        total += hs[0].shape[1]
    return finalize_pool(sums, sqs, total, layers)


def load_backbone(bb):
    if bb["kind"] == "whisper":
        m = whisper.load_model(bb["id"], device=DEVICE)
        n_blocks = len(m.encoder.blocks)
        assert max(bb["layers"]) <= n_blocks, f"{bb['name']}: layer > {n_blocks}"
        n_mels = m.dims.n_mels
        del m.decoder                      # not needed, saves memory
        m.encoder.eval()
        return {"model": m, "n_mels": n_mels}
    from transformers import AutoFeatureExtractor, AutoModel
    fe = AutoFeatureExtractor.from_pretrained(bb["id"])
    model = AutoModel.from_pretrained(bb["id"]).to(DEVICE).eval()
    assert max(bb["layers"]) <= model.config.num_hidden_layers, f"{bb['name']}: layer too large"
    return {"model": model, "fe": fe}


def extract_backbone(bb):
    cache_dir = EMB_CACHE_DIR / bb["name"]
    cache_dir.mkdir(parents=True, exist_ok=True)
    items = ([("train", f, TRAIN_AUDIO_DIR) for f in train_df["filename"].astype(str)] +
             [("test", f, TEST_AUDIO_DIR) for f in test_df["filename"].astype(str)])
    fn = whisper_layer_embedding if bb["kind"] == "whisper" else hf_layer_embedding
    state, arrs, n_new, n_fail = None, [], 0, 0

    for split, fname, adir in tqdm(items, desc=bb["name"]):
        p = cache_dir / f"{split}_{Path(fname).stem}.npy"
        if p.exists():
            arrs.append(np.load(p))
            continue
        try:
            if state is None:
                state = load_backbone(bb)
            y_audio, _ = librosa.load(str(Path(adir) / fname), sr=16000, mono=True)
            emb = fn(state, y_audio, bb["layers"])
            np.save(p, emb)
            arrs.append(emb)
            n_new += 1
        except Exception as e:
            print(f"FAILED {bb['name']} {fname}: {e}")
            arrs.append(None)
            n_fail += 1

    if state is not None:
        del state
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    shape = next(a.shape for a in arrs if a is not None)
    full = np.full((len(arrs),) + shape, np.nan, dtype=np.float32)
    for i, a in enumerate(arrs):
        if a is not None:
            full[i] = a.astype(np.float32)
    n_tr = len(train_df)
    tr, te = full[:n_tr], full[n_tr:]
    mu = np.nanmean(tr, axis=0)                   # fill failed clips with the training mean
    tr = np.where(np.isnan(tr), mu, tr)
    te = np.where(np.isnan(te), mu, te)
    print(f"{bb['name']}: shape {tr.shape} | newly extracted {n_new} | failed {n_fail}")
    return tr, te

### 10b. Run the extraction for every enabled backbone

**What the cell does**
Loops over `BACKBONES`; for each enabled one calls `extract_backbone` and stores the result in the `EMB` dictionary as
`EMB[name] = {"tr": (769, n_layers, 2*dim), "te": (216, n_layers, 2*dim), "layers": [...]}`.

**What to look for**
* The first run downloads each model (Whisper-medium about 1.4 GB, WavLM-large and HuBERT-large about 1.2 GB each) and is slow; progress bars show clips processed.
* A line per backbone with the array shape and `failed 0`.
* If you hit a GPU out-of-memory error, disable the largest backbone or restart the runtime and rerun; finished clips are cached.

In [ ]:
EMB = {}
for bb in BACKBONES:
    if not bb["enabled"]:
        print("skipping", bb["name"])
        continue
    tr_arr, te_arr = extract_backbone(bb)
    EMB[bb["name"]] = {"tr": tr_arr, "te": te_arr, "layers": bb["layers"]}

ACTIVE = list(EMB.keys())
print("\nActive backbones:", ACTIVE)

## 11. Adversarial validation: do train and test look alike?

**What the cell does**
Trains a simple classifier to tell **training clips from test clips** using audio embeddings (standardised, reduced to 30 PCA components) and, separately, the tabular features. It reports the cross-validated **AUC**.

**How to read it**
* AUC about 0.50: train and test are indistinguishable (what you want).
* AUC 0.60 - 0.70: mild difference.
* AUC above about 0.70: the test audio differs systematically (recording device, noise, speaker mix). Models validated with ordinary CV will then be less reliable on the leaderboard, and robust choices (Ridge, fusion, averaging) become more important than squeezing CV.

**Why here**
In v3 the test predictions had a standard deviation of 0.77 while the labels have 1.24. That can be normal shrinkage, or it can be a sign of a shift. This test helps tell the two apart.

In [ ]:
def adversarial_auc(Z_train, Z_test, n_components=30, seed=SEED):
    Z = np.vstack([Z_train, Z_test])
    Z = StandardScaler().fit_transform(Z)
    Z = PCA(n_components=min(n_components, Z.shape[1], Z.shape[0] - 1), random_state=seed).fit_transform(Z)
    lab = np.r_[np.zeros(len(Z_train)), np.ones(len(Z_test))]
    clf = LogisticRegression(max_iter=2000, C=0.5)
    proba = cross_val_predict(clf, Z, lab, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                              method="predict_proba")[:, 1]
    return roc_auc_score(lab, proba)


for name in ACTIVE:
    mid = len(EMB[name]["layers"]) // 2
    auc = adversarial_auc(EMB[name]["tr"][:, mid, :], EMB[name]["te"][:, mid, :])
    print(f"{name:<18} (layer {EMB[name]['layers'][mid]:>2}) train-vs-test AUC = {auc:.3f}")

auc_tab = adversarial_auc(X_tab, X_tab_te)
print(f"{'tabular features':<18}              train-vs-test AUC = {auc_tab:.3f}")

## 12. Cross-validation setup

**What the cell does**
* Bins the target into five bands and builds **repeated stratified 5-fold x 3** splits (15 fits per model), the same scheme as v3, so every model is scored on identical folds and the numbers are comparable to your earlier results.
* Defines the helpers:
  * `rmse`, `evaluate` - score a prediction vector (RMSE is the competition metric; MAE, Pearson and Spearman are printed for context).
  * `run_cv(make_model, Xtr, Xte, y, splits)` - trains one model per fold, collects **out-of-fold (OOF)** predictions (averaged over the three repeats) and also **predicts the test set with every fold model and averages** those predictions. This keeps the evaluated model and the submitted model identical.
* Stores the label range for clipping predictions.

**What to look for**
`Target band counts` and `Using RepeatedStratifiedKFold`. The "RMSE of predicting the mean" is the baseline: any useful model must be far below it (1.24 in your data).

In [ ]:
y_bins = pd.cut(y, bins=[-np.inf, 1.5, 2.5, 3.5, 4.5, np.inf], labels=False).astype(int)
bin_counts = np.bincount(y_bins)
print("Target band counts:", bin_counts.tolist())

if bin_counts[bin_counts > 0].min() >= N_SPLITS:
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    splits = list(cv.split(X_tab, y_bins))
    print("Using RepeatedStratifiedKFold")
else:
    cv = RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    splits = list(cv.split(X_tab))
    print("Rare target band -> using RepeatedKFold")
print("Fold fits per model:", len(splits))


def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def evaluate(y_true, pred, name, verbose=True):
    r = {"model": name, "RMSE": rmse(y_true, pred), "MAE": float(mean_absolute_error(y_true, pred)),
         "Pearson": float(pearsonr(y_true, pred)[0]), "Spearman": float(spearmanr(y_true, pred)[0])}
    if verbose:
        print(f"{name:<26} RMSE {r['RMSE']:.4f} | MAE {r['MAE']:.4f} | Pearson {r['Pearson']:.4f} | Spearman {r['Spearman']:.4f}")
    return r


def run_cv(make_model, Xtr, Xte, y, splits):
    n = len(y)
    oof, cnt, te = np.zeros(n), np.zeros(n), np.zeros(len(Xte))
    for tr, va in splits:
        m = make_model()
        m.fit(Xtr[tr], y[tr])
        oof[va] += m.predict(Xtr[va])
        cnt[va] += 1
        te += m.predict(Xte)
    return oof / cnt, te / len(splits)


Y_MIN, Y_MAX = float(y.min()), float(y.max())
print("Label range:", Y_MIN, "-", Y_MAX, "| baseline RMSE of predicting the mean:", round(float(y.std()), 4))

## 13. Model factories and the tabular / text base models

**What the cell does**
* Defines reusable model builders:
  * `make_xgb`, `make_et` - tree models for the tabular features (same settings as v3).
  * `make_ridge(lo, hi)` - standardise, then Ridge with the penalty chosen automatically by efficient leave-one-out CV over `10^lo ... 10^hi`.
  * `make_svr` - standardise, then RBF-kernel SVR with `C` chosen by an inner 3-fold search.
  * `make_ridge_raw` / `make_svr_raw` - the same without standardising, for the **fusion** matrices (which are already scaled in section 15).
* Runs the **tabular and text** base models on the shared folds: XGBoost, ExtraTrees, Ridge on the tabular features and Ridge on transcript embeddings. TF-IDF is dropped (weight 0 in your v3 stack).

**What to look for**
Results close to v3: `xgb_tab` about 0.61, `et_tab` about 0.64, `ridge_tab` about 0.66, `ridge_textemb` about 0.94. If they differ a lot, something changed in the cache or features.

In [ ]:
def make_xgb():
    return XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.8,
                        colsample_bytree=0.6, min_child_weight=3, reg_lambda=5.0,
                        objective="reg:squarederror", random_state=SEED, n_jobs=-1)


def make_et():
    return ExtraTreesRegressor(n_estimators=500, max_features=0.5, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1)


def make_ridge(lo=0, hi=5):
    return Pipeline([("sc", StandardScaler()), ("m", RidgeCV(alphas=np.logspace(lo, hi, 16)))])


def make_svr():
    return GridSearchCV(
        Pipeline([("sc", StandardScaler()), ("m", SVR(kernel="rbf", epsilon=0.1))]),
        {"m__C": [1, 3, 10]}, cv=3, scoring="neg_root_mean_squared_error")


def make_ridge_raw(lo=-2, hi=3):
    return RidgeCV(alphas=np.logspace(lo, hi, 16))


def make_svr_raw():
    return GridSearchCV(SVR(kernel="rbf", epsilon=0.1), {"C": [1, 3, 10, 30]}, cv=3,
                        scoring="neg_root_mean_squared_error")


oof_preds, test_preds, score_rows = {}, {}, []

base_experiments = [
    ("xgb_tab",       make_xgb,                  X_tab, X_tab_te),
    ("et_tab",        make_et,                   X_tab, X_tab_te),
    ("ridge_tab",     lambda: make_ridge(-1, 4), X_tab, X_tab_te),
    ("ridge_textemb", lambda: make_ridge(0, 4),  E_tx,  E_tx_te),
]
for name, factory, Xtr_, Xte_ in tqdm(base_experiments, desc="tabular/text models"):
    oof_preds[name], test_preds[name] = run_cv(factory, Xtr_, Xte_, y, splits)
    score_rows.append(evaluate(y, np.clip(oof_preds[name], Y_MIN, Y_MAX), name))

## 14. Layer sweep: which layer of which audio model is best?

**What the cell does**
For **every layer of every backbone** it trains one Ridge model on that layer's mean+std embedding (standardised, penalty chosen by LOO-CV) and scores it with the shared CV folds. Then:
* prints a **sorted table** of all layer models, so you can see which backbones and depths work;
* for each backbone keeps the **top `TOP_K_LAYERS` layers** (by CV RMSE) in `SELECTED[backbone] = [layer, ...]`;
* stores the OOF and test predictions of the selected layers in `oof_preds` / `test_preds` for the stack.

**Why Ridge for the sweep**
It is fast, has almost no hyper-parameters, and was already as good as SVR in v3 (0.566 vs 0.573), so it is a fair way to rank layers.

**What to look for**
* Which backbone is best, and whether the best layers are in the middle or at the end. Expect middle layers to beat the last layer for the speech-SSL models.
* Compare the best entries to v3's `ridge_whisper` (0.5656). Anything clearly below that is a real gain.
* A mild selection bias exists (we pick the best of many by the same CV), which is why the final stack is judged against several models and not only the single best layer.

In [ ]:
sweep_rows, sweep_oof, sweep_te = [], {}, {}

for name in ACTIVE:
    for li, layer in enumerate(EMB[name]["layers"]):
        key = f"{name}_L{layer}"
        Xtr_ = EMB[name]["tr"][:, li, :].astype(np.float64)
        Xte_ = EMB[name]["te"][:, li, :].astype(np.float64)
        oof, te = run_cv(lambda: make_ridge(1, 5.5), Xtr_, Xte_, y, splits)
        sweep_oof[key], sweep_te[key] = oof, te
        r = evaluate(y, np.clip(oof, Y_MIN, Y_MAX), key, verbose=False)
        r.update({"backbone": name, "layer": layer})
        sweep_rows.append(r)
        print(f"{key:<24} RMSE {r['RMSE']:.4f}")

sweep_df = pd.DataFrame(sweep_rows).sort_values("RMSE").reset_index(drop=True)
print("\n=== Layer sweep (best first) ===")
display(sweep_df[["model", "backbone", "layer", "RMSE", "MAE", "Pearson", "Spearman"]])
print("Reference: v3 ridge_whisper (last layer, whisper-small) = 0.5656")

SELECTED = {}
for name in ACTIVE:
    sub = sweep_df[sweep_df["backbone"] == name].head(TOP_K_LAYERS)
    SELECTED[name] = sub["layer"].tolist()
    for _, row in sub.iterrows():
        key = row["model"]
        oof_preds[f"ridge_{key}"], test_preds[f"ridge_{key}"] = sweep_oof[key], sweep_te[key]
        score_rows.append({k: row[k] for k in ["model", "RMSE", "MAE", "Pearson", "Spearman"]} | {"model": f"ridge_{key}"})
print("\nSelected layers per backbone:", SELECTED)

## 15. SVR on the selected layers, and fusion models

### 15a. SVR on each selected layer
RBF-kernel SVR captures non-linear structure that Ridge cannot, and its errors differ from Ridge's, so it adds diversity. In v3 it was the stacker's top model. Here it is run only on the layers chosen in section 14 to keep runtime reasonable.

### 15b. Fusion models (combine the audio models *before* predicting)
The stacker can only combine finished predictions. A fusion model sees the embeddings of **all backbones at once** and can find structure that a single backbone does not show.

* **Block scaling:** each block (one layer of one backbone, the tabular features, or the text embedding) is standardised using training statistics and divided by `sqrt(dim)`. Every block then has the **same total variance (1)**, so a 2048-dimensional audio block does not drown a 132-dimensional tabular block.
* `fusion_audio` = best layer of every audio backbone.
* `fusion_all` = `fusion_audio` + tabular features + transcript embedding.
* Each fusion matrix is used by a Ridge and an SVR model.

**What to look for**
Fusion RMSE should be lower than the best single layer from the sweep. If `fusion_all` is better than `fusion_audio`, the grammar-specific tabular features still add information beyond the audio.

In [ ]:
# ---------- 15a. SVR on the selected layers ----------
for name in ACTIVE:
    for layer in SELECTED[name]:
        li = EMB[name]["layers"].index(layer)
        key = f"svr_{name}_L{layer}"
        Xtr_ = EMB[name]["tr"][:, li, :].astype(np.float64)
        Xte_ = EMB[name]["te"][:, li, :].astype(np.float64)
        oof_preds[key], test_preds[key] = run_cv(make_svr, Xtr_, Xte_, y, splits)
        score_rows.append(evaluate(y, np.clip(oof_preds[key], Y_MIN, Y_MAX), key))


# ---------- 15b. fusion models ----------
def block_scale(tr, te):
    sc = StandardScaler().fit(tr)
    f = 1.0 / np.sqrt(tr.shape[1])
    return sc.transform(tr) * f, sc.transform(te) * f


audio_blocks_tr, audio_blocks_te = [], []
for name in ACTIVE:
    best_layer = SELECTED[name][0]
    li = EMB[name]["layers"].index(best_layer)
    a, b = block_scale(EMB[name]["tr"][:, li, :].astype(np.float64), EMB[name]["te"][:, li, :].astype(np.float64))
    audio_blocks_tr.append(a)
    audio_blocks_te.append(b)

tab_a, tab_b = block_scale(X_tab, X_tab_te)
txt_a, txt_b = block_scale(E_tx, E_tx_te)

X_fus_audio = np.hstack(audio_blocks_tr)
X_fus_audio_te = np.hstack(audio_blocks_te)
X_fus_all = np.hstack(audio_blocks_tr + [tab_a, txt_a])
X_fus_all_te = np.hstack(audio_blocks_te + [tab_b, txt_b])
print("Fusion matrices:", X_fus_audio.shape, X_fus_all.shape)

fusion_experiments = [
    ("ridge_fusion_audio", make_ridge_raw, X_fus_audio, X_fus_audio_te),
    ("svr_fusion_audio",   make_svr_raw,   X_fus_audio, X_fus_audio_te),
    ("ridge_fusion_all",   make_ridge_raw, X_fus_all,   X_fus_all_te),
    ("svr_fusion_all",     make_svr_raw,   X_fus_all,   X_fus_all_te),
]
for name, factory, Xtr_, Xte_ in fusion_experiments:
    oof_preds[name], test_preds[name] = run_cv(factory, Xtr_, Xte_, y, splits)
    score_rows.append(evaluate(y, np.clip(oof_preds[name], Y_MIN, Y_MAX), name))

## 16. Stacking, comparison with v3, and optional calibration

**What the cell does**
1. **Stacks** every model's out-of-fold predictions with a **non-negative linear regression** (with intercept). Non-negativity stops the stacker from using large opposite-sign weights to fit noise. The intercept and weight sum let it correct the shrinkage that averaging introduces.
2. Reports a **stacking CV** with repeated 5-fold on the OOF matrix, next to the **simple average** and the **best single model**, and against your v3 result (`V3_STACK_CV = 0.5275`).
   This estimate is slightly optimistic (the base OOF predictions were produced on the same data), so use it for *comparing* versions, not as an absolute leaderboard forecast.
3. Prints the **stacker weights** so you can see which families matter.
4. **Isotonic calibration check:** tests whether a monotone recalibration of the stacked predictions lowers CV RMSE by at least 0.003. It is only applied if it does.
5. Compares the **distribution of OOF predictions and test predictions** (mean and standard deviation) against the label distribution.

**What to look for**
* `stacked (CV)` clearly below 0.5275.
* Weights concentrated on the new audio / fusion models.
* `Test pred std` versus `OOF pred std`: a big gap together with a high adversarial AUC in section 11 suggests a train/test shift.

In [ ]:
stack_names = list(oof_preds.keys())
oof_mat = np.column_stack([oof_preds[n] for n in stack_names])
te_mat = np.column_stack([test_preds[n] for n in stack_names])
print(f"Stacking {len(stack_names)} models")

summary_all = pd.DataFrame(score_rows).drop_duplicates("model").sort_values("RMSE").reset_index(drop=True)
best_single = summary_all.iloc[0]

# simple average of the 5 best models
top5 = summary_all["model"].head(5).tolist()
avg_top5 = np.column_stack([oof_preds[n] for n in top5]).mean(axis=1)
evaluate(y, np.clip(avg_top5, Y_MIN, Y_MAX), "average of top-5")

# stacker CV (repeated)
stack_cv = np.zeros(len(y)); cnt = np.zeros(len(y))
for tr, va in RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED).split(oof_mat):
    s = LinearRegression(positive=True).fit(oof_mat[tr], y[tr])
    stack_cv[va] += s.predict(oof_mat[va]); cnt[va] += 1
stack_cv /= cnt
stack_score = evaluate(y, np.clip(stack_cv, Y_MIN, Y_MAX), "stacked (CV)")

print(f"\nBest single model : {best_single['model']}  RMSE {best_single['RMSE']:.4f}")
print(f"v3 stacked CV     : {V3_STACK_CV:.4f}")
print(f"v4 stacked CV     : {stack_score['RMSE']:.4f}   (change {stack_score['RMSE'] - V3_STACK_CV:+.4f})")

# final stacker
stacker = LinearRegression(positive=True).fit(oof_mat, y)
print("\nStacker weights (non-zero):")
for n, w in sorted(zip(stack_names, stacker.coef_), key=lambda t: -t[1]):
    if w > 1e-6:
        print(f"  {n:<28} {w:.3f}")
print(f"  intercept                    {float(stacker.intercept_):.3f} | weight sum {stacker.coef_.sum():.3f}")

# optional isotonic calibration, kept only if it clearly helps
iso_cv = np.zeros(len(y))
for tr, va in KFold(5, shuffle=True, random_state=SEED).split(stack_cv):
    iso = IsotonicRegression(out_of_bounds="clip").fit(stack_cv[tr], y[tr])
    iso_cv[va] = iso.predict(stack_cv[va])
iso_rmse = rmse(y, np.clip(iso_cv, Y_MIN, Y_MAX))
USE_ISOTONIC = iso_rmse < stack_score["RMSE"] - 0.003
print(f"\nIsotonic calibration CV RMSE: {iso_rmse:.4f} vs {stack_score['RMSE']:.4f} -> use it: {USE_ISOTONIC}")

test_pred_raw = stacker.predict(te_mat)
if USE_ISOTONIC:
    iso_final = IsotonicRegression(out_of_bounds="clip").fit(stack_cv, y)
    test_pred_final = np.clip(iso_final.predict(test_pred_raw), Y_MIN, Y_MAX)
    oof_final = np.clip(iso_cv, Y_MIN, Y_MAX)
else:
    test_pred_final = np.clip(test_pred_raw, Y_MIN, Y_MAX)
    oof_final = np.clip(stack_cv, Y_MIN, Y_MAX)

print("\nLabel      mean/std:", round(float(y.mean()), 3), round(float(y.std()), 3))
print("OOF pred   mean/std:", round(float(oof_final.mean()), 3), round(float(oof_final.std()), 3))
print("Test pred  mean/std:", round(float(test_pred_final.mean()), 3), round(float(test_pred_final.std()), 3))
display(summary_all.head(15))

## 17. Error analysis by label band

**What the cell does**
Splits the training set by true score band and shows, for the final out-of-fold predictions, the **RMSE and the bias** (mean prediction minus mean truth) in each band.

**How to read it**
* Regression models tend to **over-predict low scores and under-predict high scores** (shrinkage toward the mean). A strongly positive bias in the lowest band and a negative bias in the highest band confirms it.
* Your training set has only 41 clips with scores up to 1.5, and v3's smallest test prediction was 1.75, so the model rarely predicts very low scores. If the lowest band dominates the error, the next improvements should target low-score recognition (for example a classifier for "very poor" clips, or weighting low scores more).
* Bands with large RMSE are where extra modeling effort pays off most.

**What to look for**
The band with the highest RMSE and the sign of the bias per band.

In [ ]:
bands = pd.cut(y, bins=[-0.01, 1.5, 2.5, 3.5, 4.5, 5.01],
               labels=["0-1.5", "2-2.5", "3-3.5", "4-4.5", "5"])
rows = []
for b in bands.categories:
    m = (bands == b)
    if m.sum() == 0:
        continue
    rows.append({"band": b, "n": int(m.sum()),
                 "RMSE": rmse(y[m], oof_final[m]),
                 "bias (pred - true)": float((oof_final[m] - y[m]).mean())})
band_df = pd.DataFrame(rows)
display(band_df)
print("Overall OOF RMSE:", round(rmse(y, oof_final), 4))

## 18. Final submission with safe ID reconciliation

**What the cell does**
Writes the submission files and prevents the silent-zero problem from v3:

1. **Always** writes `submission_v4_all_test.csv`: one row for each of the 216 files in `test.csv`, using the exact `filename` strings from `test.csv`.
2. Tries to align to `sample_submission.csv`:
   * First by **exact** ID, then by **normalised** ID (extension removed, lower-case).
   * If at least **95%** of sample IDs are matched, it writes `submission_v4.csv` with exactly the sample's rows and order. Unmatched rows (if any) get the **training mean** (never 0), and the count is printed.
   * If fewer than 95% match, it does **not** create a misleading sample-aligned file and prints a warning with examples. In that case look at the ID examples from section 5, because the sample file and `test.csv` probably describe different sets, and you should upload the file whose IDs match what Kaggle expects.
3. Runs sanity assertions: no NaN, predictions inside the label range, correct row counts.

**What to look for**
* The line `matched ... / ...` and which file to upload.
* The description statistics: the mean should be near the training mean (3.3), **not** about 0.4 as in the v3 sample-aligned file.

In [ ]:
# ---- file with every test.csv row (always written) ----
all_sub = pd.DataFrame({id_col: test_df["filename"].astype(str).values, label_col: test_pred_final})
assert len(all_sub) == len(test_df)
assert all_sub[label_col].notna().all(), "NaN predictions"
assert all_sub[label_col].between(Y_MIN, Y_MAX).all(), "Out-of-range predictions"
all_sub.to_csv("/content/submission_v4_all_test.csv", index=False)
all_sub.to_csv(DATA_ROOT / "submission_v4_all_test.csv", index=False)
print(f"Saved submission_v4_all_test.csv ({len(all_sub)} rows)")

# ---- file aligned to sample_submission ----
exact_map = dict(zip(test_df["filename"].astype(str), test_pred_final))
norm_map = {norm_id(k): v for k, v in exact_map.items()}
fallback = float(y.mean())

values, n_exact, n_norm, n_missing, missing_examples = [], 0, 0, 0, []
for sid in sample_sub[id_col].astype(str):
    if sid in exact_map:
        values.append(exact_map[sid]); n_exact += 1
    elif norm_id(sid) in norm_map:
        values.append(norm_map[norm_id(sid)]); n_norm += 1
    else:
        values.append(fallback); n_missing += 1
        if len(missing_examples) < 5:
            missing_examples.append(sid)

n_sample = len(sample_sub)
print(f"\nSample rows: {n_sample} | matched exact: {n_exact} | matched normalised: {n_norm} | unmatched: {n_missing}")

if (n_exact + n_norm) / n_sample >= 0.95:
    aligned = pd.DataFrame({id_col: sample_sub[id_col].astype(str).values, label_col: np.clip(values, Y_MIN, Y_MAX)})
    assert len(aligned) == n_sample and aligned[label_col].notna().all()
    aligned.to_csv("/content/submission_v4.csv", index=False)
    aligned.to_csv(DATA_ROOT / "submission_v4.csv", index=False)
    print("Saved submission_v4.csv aligned to sample_submission  <-- upload this one")
    print(aligned[label_col].describe())
else:
    print("WARNING: fewer than 95% of sample_submission IDs match test.csv, NOT writing a sample-aligned file.")
    print("Unmatched examples:", missing_examples)
    print("Compare with test.csv IDs:", test_ids_raw[:5])
    print("Upload submission_v4_all_test.csv or fix the ID mapping first.")
    print(all_sub[label_col].describe())

print("\nFailed test files (imputed):", [f for f, _ in test_failed])

## 19. Save everything needed to continue later

**What the cell does**
Pickles the OOF and test predictions of every model, the stacker, the selected layers, the layer-sweep table and the config into `pipeline_v4.pkl` on Drive, then tries to download the main submission file.

**Why**
You can reload the OOF predictions later and test new stacking ideas or add a new model in seconds, without rerunning feature extraction or cross-validation.

In [ ]:
with open(DATA_ROOT / "pipeline_v4.pkl", "wb") as fh:
    pickle.dump({
        "config": {"backbones": BACKBONES, "top_k": TOP_K_LAYERS, "seed": SEED, "whisper_size": WHISPER_SIZE},
        "feature_columns": FEATURE_COLUMNS,
        "selected_layers": SELECTED,
        "stack_names": stack_names,
        "stacker": stacker,
        "use_isotonic": USE_ISOTONIC,
        "label_range": (Y_MIN, Y_MAX),
        "oof_preds": oof_preds,
        "test_preds": test_preds,
        "sweep_table": sweep_df,
        "summary": summary_all,
    }, fh)
print("Saved pipeline_v4.pkl")

try:
    from google.colab import files
    import os as _os
    path = "/content/submission_v4.csv" if _os.path.exists("/content/submission_v4.csv") else "/content/submission_v4_all_test.csv"
    files.download(path)
except Exception as e:
    print("Download skipped:", e)

## 20. If you still want more (in order of expected value)

1. **Read sections 14, 16 and 17 first.** The layer sweep tells you which backbone and depth work. Add that family's neighbours (more layers, larger sibling models such as `whisper large-v3`, `wav2vec2-large-xlsr-53`) to `BACKBONES`.
2. **Train a small neural head on frame-level features** instead of mean/std pooling: attention pooling over the frames of the best layer, a few epochs, 5-fold. Pooling throws away temporal structure (where the pauses and errors are); a learned head can use it. This is the usual next step in speech-assessment work, but it needs frame-level features stored (about 2 GB for one layer) and more code.
3. **Fine-tune a speech encoder end-to-end** (for example Whisper-small encoder or WavLM-base with a regression head). On 769 clips it needs low learning rates, early stopping, several seeds and probably a larger GPU, but it is how top solutions to tasks like this are usually built.
4. **Target the low scores.** If section 17 shows large errors for scores up to 1.5, add a binary "very low score" classifier or sample weights.
5. **More seeds / repeats** (`N_REPEATS = 5`) for a smoother OOF matrix and test predictions.
6. **Check the ID issue.** If section 5 shows the sample submission differs from `test.csv`, make sure the file you upload matches what Kaggle expects.

A last reminder: CV and leaderboard differ in your case (0.53 vs 0.39), so use the CV *difference* between versions as the guide and treat the leaderboard as the final check.